# ICS 604: APPLIED DATA SCIENCE

## Kernel Density Estimation (KDE)

---

In [ ]:
import numpy as np
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt

## Introduction to Kernel Density Estimation: Histograms

A histogram is one of the most intuitive ways to visualize the distribution of numerical data. Suppose we generate a set of random integers using a function such as `randint(0, 100)`. To construct a histogram, we divide the range of possible values into equal-sized intervals, or *bins*, and then count how many data points fall into each interval. If we choose a bin width of 10, the bins would be 0–9, 10–19, 20–29, and so on up to 90–99. By tallying the number of observations within each bin, we obtain a frequency distribution that provides a coarse but useful approximation of the underlying probability density.

Rather than manually implementing the counting process using Python’s standard library, we can rely on `numpy`, which provides efficient and concise tools for numerical computation. With functions such as `numpy.histogram`, we can automatically compute the counts for each bin and, if desired, normalize them to approximate a probability density. This histogram representation serves as a foundational step toward more advanced density estimation techniques, such as kernel density estimation, which smooths the distribution to produce a continuous curve.

In [ ]:
random_draws = np.random.randint(0, 100, 1000)
random_draws[0:15]

In [ ]:
bins = np.arange(0, 101, 10)
bins

In [ ]:
# Counts using numpy.histogram()

counts_and_bins = np.histogram(random_draws, bins)
print(counts_and_bins, type(counts_and_bins))
print("=+=" * 15)
print(counts_and_bins[0])
print(counts_and_bins[1])

In [ ]:
# Align x-coordinate at the left edge of the bar

plt.figure(figsize=(8, 4))
plt.bar(counts_and_bins[1][:-1], counts_and_bins[0], edgecolor='k', width=9, align='edge')
plt.show()

### Frequencies and Probability Distributions

When we construct a histogram by counting how many observations fall into each bin, we obtain frequencies. However, these raw counts do not constitute a valid probability distribution.

**Why not?**

A valid probability distribution must satisfy two key properties:

1. All probabilities must be non-negative.
2. The probabilities must sum to 1.

While histogram frequencies are non-negative, they generally do not sum to 1 — they sum to the total number of observations $n$. Therefore, the histogram counts alone are not probabilities; they are simply totals.

**How can we turn it into a probability distribution?**

To convert frequencies into a valid probability distribution, we normalize them. This is done by dividing each bin count by the total number of observations:

$$
\text{Probability of bin }i = \frac{\text{Count in bin }i}{\text{Total counts}}
$$


After this transformation:
- Each value is between 0 and 1
- The probabilities across all bins sum to 1

These normalized values are called **relative frequencies**, and together they form a valid discrete probability distribution over the bins.

If we further divide by the bin width, we obtain a **density**, which is useful when approximating a continuous probability distribution — an important step toward kernel density estimation.
  

In [ ]:
normalized_counts = counts_and_bins[0] / sum(counts_and_bins[0])

plt.figure(figsize=(8, 4))
plt.bar(counts_and_bins[1][:-1], normalized_counts, width=9, align='edge')
plt.show()

In [ ]:
print(sum(normalized_counts))

### Nonparametric Density Estimation for Discrete Random Variables

Histograms provide a simple, nonparametric way to estimate the distribution of a random variable. Using numpy, we can compute either raw counts or normalized values directly. In particular, `np.histogram(numbers, bins, density=True)` returns normalized counts, effectively producing values similar to a probability mass function (pmf). When normalized, the total area (or total probability) sums to 1, allowing the histogram to approximate a probability distribution.

In [ ]:
# Normalized counts using numpy.histogram()

densities_and_bins = np.histogram(random_draws, bins, density=True)

plt.figure(figsize=(8, 4))
plt.bar(densities_and_bins[1][:-1], densities_and_bins[0], width=9, align='edge')
plt.show()

In [ ]:
print(sum(densities_and_bins[0]))

However, a significant limitation of histograms is their sensitivity to the choice of bin boundaries. The placement and width of bins are somewhat arbitrary decisions, yet they can strongly influence the resulting visualization. Two histograms constructed from the same dataset but with slightly different starting points or bin widths may produce noticeably different shapes.

Because observations near bin edges may fall into different intervals depending on how the bins are defined, the estimated probabilities within each bin can shift. As a result, the implied probability distribution — and thus the interpretation of the data — may change simply due to different bin boundary choices. This instability motivates the development of smoother nonparametric density estimation methods, such as kernel density estimation, which reduce sensitivity to arbitrary bin definitions.

<center><img src="https://www.dropbox.com/scl/fi/7ctypxz7ywpu6rjp98lrg/bin_diffs.png?rlkey=e1qddcfews59n91clw81xejwe&st=iznlhg7u&dl=1" width="700"></center>

<center><em>Histograms on same data: bins are of the same size, but histogram on the right has bins shifted to the right.</em></center>

In [ ]:
data = [119, 120, 121, 125, 129, 130, 130.1, 130.5, 130.7, 131]
bins_2 = range(115, 136, 5)
print(list(bins_2))

In [ ]:
counts = np.histogram(data, bins_2, density=True)
print(counts[0])

plt.figure(figsize=(8, 3))
plt.bar(x=counts[1][:-1], height=counts[0], width=4)
plt.xticks(bins_2[:-1])
plt.show()

In [ ]:
# Plotting using Seaborn

import seaborn as sns

plt.figure(figsize=(8, 3))
sns.barplot(x=counts[1][:-1], y=counts[0])
plt.show()

In [ ]:
bins_3 = range(114, 136, 5)
print(list(bins_3))

counts = np.histogram(data, bins_3, density=True)
print(counts[0])
plt.figure(figsize=(8, 3))
sns.barplot(x=counts[1][:-1], y=counts[0])
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

bins_2 = range(115, 136, 5)
counts = np.histogram(data, bins_2, density=True)
plt.subplot(1, 2, 1)
sns.barplot(x=counts[1][:-1], y=counts[0])

bins_3 = range(114, 136, 5)
counts = np.histogram(data, bins_3, density=True)
plt.subplot(1, 2, 2)
sns.barplot(x=counts[1][:-1], y=counts[0]);

### Estimating Densities Using Kernels

One way to bypass the problem of arbitrary bin boundaries is to eliminate bins altogether. Instead of grouping observations into fixed intervals, we allow each data point to contribute directly to the density estimate. For example, we can place a small **uniform block** (a rectangle of fixed width and height) centered at every observed data point.
    
<center><img src="https://www.dropbox.com/scl/fi/2oyugr02cjenbnxhmnzwr/top_hat_1.png?rlkey=54xrtbimbxw7ksrnqius3km81&st=uwzierao&dl=1" width="600" height="300"></center>
<center><em>Each data point contributes a uniform <strong>height</strong> value. The shaded regions representing two or more overlapping blocks</em></center>
<br>
To estimate the density at any position along the $x$-axis — including locations where no observations were recorded — we compute the sum of the heights of all blocks that overlap that position. In other words, the density at a point is determined by how many nearby observations contribute mass there. This produces a single, consistent visualization that does not depend on arbitrary bin starting points.

<center><img src="https://www.dropbox.com/scl/fi/s06kebkd5i1w6txl1qxj4/top_hat_2.png?rlkey=vweqn9u3rhj9kn2wvyzezf8hi&st=ilf8xmu0&dl=1" width="600" height="300"></center>

<br>
Because each block is centered directly on a data point, the procedure is deterministic and independent of bin alignment. Additionally, this method assigns nonzero density even in regions where no sample was directly observed (as long as nearby points contribute mass). This is desirable, since real-world samples are finite and often sparse in parts of the sample space. Naturally, regions with many nearby observations produce taller peaks, reflecting higher estimated density. In this way, we can use a finite data sample to make inferences about the underlying population distribution.<br><br>

The square block used in this construction is called a **kernel**, and the overall method is known as **Kernel Density Estimation (KDE)**. However, using square (uniform) kernels often leads to a jagged or rough density curve. Small differences in nearby data points can produce noticeable jumps in the estimated density. This raises an important question: should very close values really generate such abrupt changes in density?

<center><img src="https://www.dropbox.com/scl/fi/54p4joefqq5mjvq8ve2q8/top_hat_example.png?rlkey=jfg2thl4h5ff23go36zfqmwkv&st=pf2jh9i9&dl=1" width="400"></center>

<br>
This concern motivates the use of smoother kernels (such as Gaussian kernels), which produce more stable and realistic density estimates.

### Using Different Kernels

In practice, we are accustomed to probability densities being smooth. The jagged shape produced by a square (uniform) kernel can therefore feel unnatural, especially when modeling continuous phenomena. This motivates replacing the square block with a smoother curve centered at each data point.

A natural candidate is the Gaussian kernel. For each observed data point $x'$ we place a Gaussian-shaped curve centered at that point with a chosen scale (bandwidth) $\sigma$ — for example, $\sigma=0.1$. Instead of contributing a constant height within a fixed interval (as in the square kernel), each data point contributes smoothly according to the value of the Gaussian kernel evaluated at the distance from that point.

<center><img src="https://www.dropbox.com/scl/fi/pok5k62fhbfpyd5eaw9gs/gaussian_kernel.png?rlkey=p8akzu2q4xnwtywf7gpl7mdil&st=rd3auisy&dl=1" width="300"></center>

To estimate the density at a position $x$, we sum the contributions from all Gaussian kernels evaluated at $x$. That is, each data point contributes proportionally to how close $x$ is to its center — nearby points contribute more, distant points contribute less.

Formally, the contribution from a data point $x'$ at position $x$ is:

$$ 
\LARGE\frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(x-x')^2}{2\sigma^2}}
$$

where $\sigma$ controls the smoothness (bandwidth). In Python, this contribution can be computed using: norm.pdf($x$, $x'$, $\sigma$).

By averaging these smooth Gaussian curves, we obtain a continuous and smooth density estimate derived directly from the data.

$$ 
\LARGE
\hat{f}(x) = \frac{1}{n} \sum_{x' \in X} \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(x-x')^2}{2\sigma^2}}
$$
where $n$ is the number of data points and $\sigma$ is the bandwidth.

The resulting curve avoids the abrupt jumps of square kernels and produces a more realistic approximation of the underlying distribution.

<center><img src="https://www.dropbox.com/scl/fi/9e7t13vkf5ep7iovck3q0/final_kernel.png?rlkey=k1fk57ou1k452885enmkhb8t9&st=xlak2cgo&dl=1" width="700"></center>

### KDE Kernels

While the Gaussian kernel is the most commonly used choice in Kernel Density Estimation (KDE), it is not the only option. A kernel is a symmetric, non-negative function that integrates to 1 and determines how individual data points contribute to the overall density estimate. Many different kernel functions are available, each with slightly different shapes and properties.

Some commonly used kernels include:

- Gaussian kernel – assigns weight according to the normal distribution and has infinite support.
- Uniform (Rectangular/Boxcar/Tophat) kernel – assigns equal weight within a fixed window and zero outside.
- Epanechnikov kernel – gives a parabolic shape and is optimal in a mean squared error sense among bounded kernels.
- Exponential kernel – assigns weight that decays exponentially with distance from the center, emphasizing nearby points while giving a long tail for distant points.
- Linear (Triangular) kernel – assigns linearly decreasing weight as distance from the center increases.
- Cosine kernel – a smooth, bell-shaped kernel based on the cosine function, giving more weight to points near the center and tapering off to zero at the edges.

<center><img src="https://www.dropbox.com/scl/fi/6dim7io0kvcqingieyfzc/kernels.png?rlkey=0jk1y1y7n30mgnrmvtbswm9fe&st=wfaslmq6&dl=1" width="600"></center>

<br>
Although these kernels differ in shape, the choice of bandwidth (the scale or smoothing parameter) is generally more important than the kernel type. The bandwidth determines how wide each kernel is and therefore controls the bias–variance tradeoff:

- Small bandwidth $\rightarrow$ very detailed, possibly noisy estimate (high variance).
- Large bandwidth $\rightarrow$ very smooth estimate, possibly oversmoothed (high bias).

In practice, the Gaussian kernel is popular because it produces smooth curves and is mathematically convenient. However, many kernels produce very similar results when the bandwidth is appropriately chosen.

### Differences Between the Kernels 

Different kernels in KDE influence the estimated density in distinct ways because each kernel emphasizes the contribution of neighboring points differently. For example, a uniform (tophat) kernel treats all points within its width equally, while a Gaussian or exponential kernel assigns higher weight to points closer to the center and gradually decreases the weight for more distant points. Cosine and triangular kernels provide intermediate smoothing behaviors, emphasizing nearby points more than distant ones but with different decay profiles.

Kernels also differ in computational efficiency. Some are faster to compute than others due to their support and functional form:

- Tophat (Uniform) kernel – since it only considers points within a fixed width, contributions can be computed efficiently, yielding a complexity of at most $O(n)$ for $n$ data points.
- Gaussian kernel – has infinite support, meaning every point potentially contributes to the density at every evaluation point. This increases computation, with complexity up to $O(n^2)$.

In practice, the choice of kernel balances computational efficiency and smoothness. For large datasets, kernels with compact support like tophat or Epanechnikov can reduce computation time while still producing reasonable density estimates.

### Using KDE on Simulated Data 

To illustrate Kernel Density Estimation (KDE) in practice, consider a dataset simulated from a Gaussian distribution with mean 0 and standard deviation 0.5:

$$
X \sim \mathcal{N}(0, 0.5)
$$

Steps to explore KDE:
1. **Generate the data**: Draw 100 random points from the Gaussian distribution above. This small sample allows us to see how KDE smooths discrete observations.
2. **Build a histogram**: Construct a histogram from the generated data. This provides a discrete, binned approximation of the distribution and serves as a baseline for comparison.
3. **Apply KDE**: Compute the kernel density estimate for the same data. By placing a smooth kernel (e.g., Gaussian) at each data point and averaging the contributions, we obtain a continuous curve that approximates the underlying distribution.

Comparison:
- The histogram is stepwise and sensitive to bin boundaries, showing coarse approximations of density.
- The KDE curve is smooth and continuous, capturing the shape of the underlying Gaussian distribution more naturally.
- Dense areas in the data produce taller peaks in the KDE, while sparse areas produce lower density values, providing a more faithful representation of the distribution than the histogram.

This example highlights how KDE can provide a smooth, interpretable estimate of the underlying density from finite data, overcoming the limitations of arbitrary histogram bins.

In [ ]:
x_mean, x_scale = 0, 0.5
x = np.random.normal(x_mean, x_scale, 100)

In [ ]:
plt.figure(figsize=(8, 4))

plt.hist(x, edgecolor='k', alpha=0.5, bins=15, density=True)
plt.xlim(-3.2, 3.2)
plt.show()

### Computing the KDE of a Dataset

Kernel Density Estimation (KDE) can be implemented using the following basic algorithm:
    
__```Algorithm```__:
```
For each position x along the x-axis:
    For each data point d in the dataset:
        Compute contribution = Kernel(x - d)
    Set y(x) = average of all contributions
```       

**How many times do we compute Kernel(x - d)?**

For a dataset of $n$ points and $m$ evaluation positions along the $x$-axis:

- Gaussian or other infinite-support kernels: we potentially evaluate the kernel for all $n$ points at each position $x$, resulting in up to $𝑛 \times m$ computations.
- Compact-support kernels (like tophat, Epanechnikov): we only evaluate the kernel for points whose kernel overlaps the current position $x$, reducing the number of computations but still potentially up to $n \times m$ in the worst case.

**How can we speed up KDE?**

1. Choose a simpler kernel – Kernels with compact support, like tophat or Epanechnikov, reduce the number of data points that contribute at each $x$, making computations faster.
2. Use efficient data structures – Spatial indexing structures like k-d trees or ball trees allow rapid querying of points within a neighborhood of $x$. Instead of checking all $n$ points, you only consider nearby points whose kernels overlap $x$, significantly reducing computation, especially for large datasets.
3. Vectorized computations – In Python, libraries like numpy or `scipy.stats.gaussian_kde` can compute contributions from all points simultaneously using array operations, which is faster than looping over each point.

By combining compact kernels with efficient nearest-neighbor search, KDE can scale to large datasets without explicitly evaluating every kernel at every point.

In [ ]:
x_values = np.arange(-3, 3, 0.1)

# compute the KDE
kde = sp.stats.gaussian_kde(x, bw_method=0.8)

# estimate kernel density for the support (x-axis)
x_densities = kde.evaluate(x_values)

print(x[0:10])
print("*" * 75)
print(x_densities[0:10])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_values, x_densities, lw=2, color='black')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_values, x_densities, lw=2, color='black')
plt.hist(x, edgecolor='k', alpha=0.5, bins=15, density=True)
plt.show()

In [ ]:
# estimate kernel density for the support (x data values)
x_densities = kde.evaluate(x)

plt.figure(figsize=(8, 4))
plt.plot(x, x_densities, '.', lw=2, color='black')
plt.hist(x, edgecolor='k', alpha=0.5, bins=15, density=True)
plt.xlim(-3.2, 3.2)
plt.show()

In [ ]:
x_mean, x_scale =  0, 0.5
y_mean, y_scale =  9, 1

x_data = np.random.normal(x_mean, x_scale, 100)
y_data = np.random.normal(y_mean, y_scale, 100)
print("The first 10 samples from the first gaussian are:\n %s \n" % x_data[0:10])
print("The first 10 samples from the second gaussian are:\n %s \n" % y_data[0:10])

In [ ]:
all_data = np.concatenate([x_data, y_data])  # list_1 + list_2  
print(len(all_data))

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=20, edgecolor="k", linewidth=1, density=True, alpha=0.5)
plt.xlim(-3, 15)
plt.show()

In [ ]:
x_values = np.arange(-10, 20, 0.25)

# compute the KDE
kde = sp.stats.gaussian_kde(all_data, bw_method=0.16)

# estimate kernel density for the support (x-axis)
densities = kde.evaluate(x_values)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=20, edgecolor="k", linewidth=1, density=True, alpha=0.2)
plt.plot(x_values, densities, lw=2, color='black')
plt.xlim(-3, 15)
plt.show()